##Transforming drivers data

In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00-common/1.environment_config

In [0]:
%run ../00-common/3.silver_helpers

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.drivers"
silver_table = f"{catalog_name}.{silver_schema}.drivers"

In [0]:
drivers_df = (
    spark.table(bronze_table)
    .filter(F.col("batch_id") == v_batch_id)
)

## Dropping the url column

In [0]:
drivers_dropped_df = (
    drivers_df
    .select(
        F.col("driverId"),
        F.col("dateOfBirth"),
        F.col("name"),
        F.col("nationality"),
        F.col("ingestion_timestamp"),
        F.col("source_file"),
        F.col("batch_id")
    )
)

display(drivers_dropped_df)

## Standardizing column names

In [0]:
drivers_renamed_df = (
    drivers_dropped_df
    .withColumnsRenamed(
        {"driverId": "driver_id",
         "dateOfBirth": "date_of_birth",
         "name": "driver_name"}
    )
)

In [0]:
display(drivers_renamed_df)

In [0]:
drivers_concatinated_df = (
    drivers_renamed_df
    .withColumn("driver_name",
                F.initcap(
                    F.concat_ws(" ", F.col("driver_name.givenName"), F.col("driver_name.familyName"))
                         )
                ) 
)

display(drivers_concatinated_df)


### Dropping null business keys

In [0]:
drivers_valid_df = (
    drivers_concatinated_df
    .filter(F.col("driver_id").isNotNull())
)

display(drivers_valid_df)

In [0]:
drivers_filtered_df = (
    drivers_valid_df
    .dropDuplicates(["driver_id"])
)

display(drivers_filtered_df)


Standardizing the colum values within `nationality`

In [0]:
drivers_final_df = (
    drivers_filtered_df
    .withColumns(
        {
            "nationality": F.initcap(F.col("nationality"))
        }
    )
)

display(drivers_final_df)

## Writing into the silver delta table

In [0]:
drivers_columns_to_update = [
    "driver_id",
    "driver_name",
    "date_of_birth",
    "nationality",
    "ingestion_timestamp",
    "source_file",
    "batch_id"
]

write_to_silver(
    drivers_final_df, 
    silver_table,
    merge_condition = "t.driver_id = s.driver_id",
    columns_to_update = drivers_columns_to_update
    )

In [0]:
spark.table(silver_table).display()